# Goalkeeper Positioning — 01: Data Preparation

End-to-end data pipeline for the project: from raw StatsBomb shots and freeze frames to the rasterized tensors that feed `DangerCNN`.

**Sections:**
1. Setup
2. EDA on shots & freeze frames
3. Build train / val / test / transfer splits
4. Rasterization
5. PyTorch `Dataset` and `DataLoader`s

This notebook replaces `src/data/build_splits.py`, `src/data/rasterize.py`, `src/data/dataset.py`, `src/training/dataloaders.py`, and the EDA script `eda_shots.py`. Run cells top-to-bottom the first time you set up the project.

**Prerequisites:** `data/raw/shots_master_df.csv` and `data/raw/freeze_master_df.csv` (produced by `notebooks/01_eda.ipynb` via `mplsoccer.Sbopen`).

## 1. Setup

In [ ]:
from __future__ import annotations

import os
import random
import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from mplsoccer import Pitch, Sbopen, VerticalPitch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

# Resolve repo root from the notebook location: notebooks/<this>.ipynb -> repo root
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SHOTS_PATH = REPO_ROOT / "data/raw/shots_master_df.csv"
FREEZE_PATH = REPO_ROOT / "data/raw/freeze_master_df.csv"
SPLITS_DIR = REPO_ROOT / "data/processed/splits"
EDA_OUT_DIR = REPO_ROOT / "results/eda"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
EDA_OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"REPO_ROOT     = {REPO_ROOT}")
print(f"SHOTS_PATH    = {SHOTS_PATH}  exists={SHOTS_PATH.exists()}")
print(f"FREEZE_PATH   = {FREEZE_PATH}  exists={FREEZE_PATH.exists()}")

## 2. EDA on shots & freeze frames

Quick characterization of the raw data before we filter and split: subtype mix, goal conversion, shot locations, xG distribution, freeze-frame coverage, and goalkeeper identifiability.

### 2.1 Load and inspect

In [ ]:
shots = pd.read_csv(SHOTS_PATH)
freeze = pd.read_csv(FREEZE_PATH)
print(f"Shots  : {shots.shape[0]:,} rows × {shots.shape[1]} cols")
print(f"Freeze : {freeze.shape[0]:,} rows × {freeze.shape[1]} cols")

print("\nShots dtypes:")
print(shots.dtypes.to_string())

missing_shots = shots.isnull().sum()
missing_shots = missing_shots[missing_shots > 0].sort_values(ascending=False)
print("\nMissing values (shots):")
print(missing_shots.to_string() if len(missing_shots) else "  none")

### 2.2 Shot subtype breakdown

In [ ]:
def pct(n, total):
    return f"{n:,} ({100 * n / total:.1f}%)" if total else "0"

total_shots = len(shots)
subtype_counts = shots["sub_type_name"].value_counts(dropna=False)

print(f"Total shots: {total_shots:,}\n")
print("Breakdown by sub_type_name:")
for name, count in subtype_counts.items():
    print(f"  {str(name):<25} {pct(count, total_shots)}")

### 2.3 Goal conversion rates

In [ ]:
shots["is_goal"] = shots["outcome_name"] == "Goal"

overall_rate = shots["is_goal"].mean()
print(f"Overall conversion rate: {overall_rate:.1%}\n")

conv_by_subtype = (
    shots.groupby("sub_type_name")["is_goal"]
    .agg(shots_n="count", goals_n="sum")
    .assign(conversion=lambda d: d["goals_n"] / d["shots_n"])
    .sort_values("shots_n", ascending=False)
)
print("Conversion rate by sub_type_name:")
print(conv_by_subtype.to_string())

### 2.4 Shot location heatmap (open play)

In [ ]:
open_play = shots[shots["sub_type_name"] == "Open Play"].copy()
print(f"Open-play shots for heatmap: {len(open_play):,}")

pitch = VerticalPitch(pitch_type="statsbomb", half=True, line_color="black", line_zorder=2)
fig, ax = pitch.draw(figsize=(6, 8))
bin_stat = pitch.bin_statistic(open_play["x"], open_play["y"], bins=(40, 40))
pcm = pitch.heatmap(bin_stat, ax=ax, cmap="Reds", edgecolors="none")
fig.colorbar(pcm, ax=ax, fraction=0.03, label="Shot count")
ax.set_title("Open-play shot locations (StatsBomb data)", fontsize=13)
fig.savefig(EDA_OUT_DIR / "shot_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

### 2.5 xG distribution

In [ ]:
xg = shots["shot_statsbomb_xg"].dropna()
print(f"xG available for {len(xg):,} / {total_shots:,} shots")
print(f"  mean={xg.mean():.4f}  median={xg.median():.4f}  max={xg.max():.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(xg, bins=60, color="steelblue", edgecolor="white", linewidth=0.3)
ax.set_xlabel("StatsBomb xG")
ax.set_ylabel("Count")
ax.set_title("Distribution of StatsBomb xG across all shots")
ax.axvline(xg.median(), color="red", linestyle="--", label=f"median={xg.median():.3f}")
ax.legend()
fig.tight_layout()
fig.savefig(EDA_OUT_DIR / "xg_distribution.png", dpi=150)
plt.show()

### 2.6 Freeze frame coverage

In [ ]:
freeze_shot_ids = freeze["id"].unique()
shots_with_freeze = shots["id"].isin(freeze_shot_ids).sum()
print(f"Shots with at least one freeze frame entry : {pct(shots_with_freeze, total_shots)}")
print(f"Shots without freeze frames                : {pct(total_shots - shots_with_freeze, total_shots)}")

players_per_shot = freeze.groupby("id").size()
print(f"\nVisible players per shot:")
print(f"  min={players_per_shot.min()}  median={players_per_shot.median():.0f}  "
      f"mean={players_per_shot.mean():.1f}  max={players_per_shot.max()}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(players_per_shot.values, bins=range(1, players_per_shot.max() + 2),
        color="steelblue", edgecolor="white", linewidth=0.3, align="left")
ax.set_xlabel("Visible players in freeze frame")
ax.set_ylabel("Number of shots")
ax.set_title("Distribution of visible players per shot freeze frame")
ax.axvline(players_per_shot.median(), color="red", linestyle="--",
           label=f"median={players_per_shot.median():.0f}")
ax.legend()
fig.tight_layout()
fig.savefig(EDA_OUT_DIR / "players_per_shot.png", dpi=150)
plt.show()

### 2.7 Goalkeeper identifiability (open-play only)

The model needs the opponent goalkeeper's position. We require `position_name == 'Goalkeeper'` and `teammate == False` in the freeze frame. Anything that doesn't satisfy that is unusable.

In [ ]:
open_play_ids = shots.loc[shots["sub_type_name"] == "Open Play", "id"]
freeze_open_play = freeze[freeze["id"].isin(open_play_ids)]

open_play_with_freeze = freeze_open_play["id"].nunique()
total_open_play = len(open_play_ids)
print(f"Open-play shots              : {total_open_play:,}")
print(f"  with freeze frame          : {pct(open_play_with_freeze, total_open_play)}")

gk_mask = (freeze_open_play["position_name"] == "Goalkeeper") & (freeze_open_play["teammate"] == False)
shots_with_gk = freeze_open_play.loc[gk_mask, "id"].nunique()
print(f"  with identifiable opp. GK : {pct(shots_with_gk, total_open_play)}")
if open_play_with_freeze:
    print(f"    (% of those with freeze) : {100 * shots_with_gk / open_play_with_freeze:.1f}%")

### 2.8 Final modeling sample

In [ ]:
gk_shot_ids = freeze_open_play.loc[gk_mask, "id"].unique()
modeling_shots = shots[shots["id"].isin(gk_shot_ids)].copy()
n_modeling = len(modeling_shots)
modeling_goal_rate = modeling_shots["is_goal"].mean()

print(f"Final sample size            : {n_modeling:,}")
print(f"Goal conversion rate         : {modeling_goal_rate:.1%}")
if "match_id" in modeling_shots.columns:
    print(f"Unique matches               : {modeling_shots['match_id'].nunique():,}")

print("\nSub-type breakdown in final sample (sanity — should be all Open Play):")
print(modeling_shots["sub_type_name"].value_counts().to_string())

## 3. Build train / val / test / transfer splits

Five manifests, all under `data/processed/splits/`:

| split | content | notes |
|---|---|---|
| `train` | men 2015/16, top-5 leagues | ~70% of main pool |
| `val` | men 2015/16, top-5 leagues | ~15%, used for HP tuning |
| `test` | men 2015/16, top-5 leagues | ~15%, run once |
| `transfer_women` | all women's competitions | held out for post-hoc generalisation check |
| `transfer_men_other` | other men's competitions/seasons | held out for post-hoc generalisation check |

Splits are at the **match level** (all shots from one match go to one split), stratified by binned per-match goal rate. This prevents leakage from temporal/tactical correlations within a match.

### 3.1 Filter shots

In [ ]:
RANDOM_SEED = 42

WOMEN_COMPETITIONS = {
    "FA Women's Super League",
    "Women's World Cup",
    "UEFA Women's Euro",
    "NWSL",
}
MEN_2015_16_TOP5 = {
    ("La Liga", "2015/2016"),
    ("Premier League", "2015/2016"),
    ("Serie A", "2015/2016"),
    ("Ligue 1", "2015/2016"),
    ("1. Bundesliga", "2015/2016"),
}

shots_full = pd.read_csv(SHOTS_PATH)
freeze_small = pd.read_csv(FREEZE_PATH, usecols=["id", "teammate", "position_name"])

op = shots_full[shots_full["sub_type_name"] == "Open Play"].copy()
print(f"  Open-play shots          : {len(op):,}")

freeze_ids = set(freeze_small["id"].unique())
op = op[op["id"].isin(freeze_ids)]
print(f"  With freeze frame        : {len(op):,}")

gk_ids = freeze_small.loc[
    (freeze_small["position_name"] == "Goalkeeper") & (freeze_small["teammate"] == False), "id"
].unique()
op = op[op["id"].isin(gk_ids)].copy()
print(f"  With identifiable GK     : {len(op):,}")

op["is_goal"] = (op["outcome_name"] == "Goal").astype(int)

### 3.2 Fetch competition metadata from StatsBomb

We need the competition name and gender for each match so we can assign tiers (men 2015/16 top-5 vs. women vs. other men).

In [ ]:
parser = Sbopen()
df_comp = parser.competition()

meta_rows = []
for _, row in df_comp.iterrows():
    try:
        m = parser.match(row["competition_id"], row["season_id"])
        m["competition_name"] = row["competition_name"]
        m["season_name"] = row["season_name"]
        m["competition_gender"] = row["competition_gender"]
        meta_rows.append(m[["match_id", "competition_name", "season_name", "competition_gender"]])
    except Exception:
        continue

meta = pd.concat(meta_rows, ignore_index=True).drop_duplicates("match_id")
op = op.merge(meta, on="match_id", how="left")

missing_meta = op["competition_name"].isna().sum()
if missing_meta:
    print(f"  WARNING: {missing_meta} shots could not be matched to competition metadata")
else:
    print("  All shots matched to competition metadata.")

### 3.3 Assign tier

In [ ]:
op["gender"] = op.apply(
    lambda r: "female"
    if r["competition_gender"] == "female" or r["competition_name"] in WOMEN_COMPETITIONS
    else "male",
    axis=1,
)

def assign_tier(row):
    if row["gender"] == "female":
        return "women"
    if (row["competition_name"], row["season_name"]) in MEN_2015_16_TOP5:
        return "men_2015_16_top5"
    return "men_tournament_other"

op["tier"] = op.apply(assign_tier, axis=1)

print("Shots by tier:")
for tier, n in op["tier"].value_counts().items():
    print(f"  {tier:<25} {n:,}")

### 3.4 Match-level stratified split

For the main pool (men 2015/16 top-5), we bin matches into goal-rate quartiles, then stratify-split match IDs 70/15/15. Shots inherit their match's split assignment.

In [ ]:
MANIFEST_COLS = ["id", "match_id", "competition_name", "season_name", "is_goal"]

main = op[op["tier"] == "men_2015_16_top5"].copy()

match_stats = (
    main.groupby("match_id")["is_goal"]
    .agg(shots_n="count", goals_n="sum")
    .assign(goal_rate=lambda d: d["goals_n"] / d["shots_n"])
    .reset_index()
)
match_stats["rate_bin"] = pd.qcut(
    match_stats["goal_rate"], q=4, labels=False, duplicates="drop"
)

train_match_ids, temp_match_ids = train_test_split(
    match_stats["match_id"],
    test_size=0.30,
    stratify=match_stats["rate_bin"],
    random_state=RANDOM_SEED,
)
temp_stats = match_stats[match_stats["match_id"].isin(temp_match_ids)]
val_match_ids, test_match_ids = train_test_split(
    temp_stats["match_id"],
    test_size=0.50,
    stratify=temp_stats["rate_bin"],
    random_state=RANDOM_SEED,
)

train_shots = main[main["match_id"].isin(train_match_ids)][MANIFEST_COLS]
val_shots   = main[main["match_id"].isin(val_match_ids)][MANIFEST_COLS]
test_shots  = main[main["match_id"].isin(test_match_ids)][MANIFEST_COLS]
transfer_women     = op[op["tier"] == "women"][MANIFEST_COLS]
transfer_men_other = op[op["tier"] == "men_tournament_other"][MANIFEST_COLS]

splits = {
    "train":              train_shots,
    "val":                val_shots,
    "test":               test_shots,
    "transfer_women":     transfer_women,
    "transfer_men_other": transfer_men_other,
}
file_map = {k: f"{k}_shot_ids.csv" for k in splits}

### 3.5 Save manifests + summary table

In [ ]:
for key, df in splits.items():
    path = SPLITS_DIR / file_map[key]
    df.to_csv(path, index=False)
    print(f"Saved {path}")

print("\n" + "=" * 70)
print(f"{'Split':<22} {'Shots':>8} {'Goals':>7} {'Rate':>7} {'Matches':>8}")
print("=" * 70)
for key, df in splits.items():
    n = len(df)
    g = int(df["is_goal"].sum())
    rate = g / n if n else 0
    m = df["match_id"].nunique()
    print(f"{key:<22} {n:>8,} {g:>7,} {rate:>7.1%} {m:>8,}")
print("=" * 70)

## 4. Rasterization

Each shot becomes a 5-channel image at 1 m / cell, cropped to the attacking half: tensor shape `(5, 80, 60)`.

| ch | content | combine |
|---|---|---|
| 0 | shooter | single Gaussian |
| 1 | ball | single Gaussian (= shooter for now; pass-end position later for headers/first-time shots) |
| 2 | attacking teammates (excl. shooter) | elementwise max of per-player Gaussians |
| 3 | defenders (excl. GK) | elementwise max |
| 4 | goalkeeper | single Gaussian |

σ = 1.5 m. **Elementwise max (not sum)** keeps values in `[0, 1]` and makes occupancy the primary signal — also keeps multi-player channels semantically comparable to the single-Gaussian ones.

Off-crop handling: a shooter outside the crop raises `ShotOutsideCropError` (filtered upstream); other off-crop players are clip-rendered.

### 4.1 Coordinate convention and constants

In [ ]:
X_MIN, X_MAX = 60.0, 120.0
Y_MIN, Y_MAX = 0.0, 80.0
GRID_W = int(X_MAX - X_MIN)   # 60
GRID_H = int(Y_MAX - Y_MIN)   # 80
SIGMA = 1.5
GOAL_CENTER = (120.0, 40.0)


class ShotOutsideCropError(ValueError):
    """Raised when the shooter's position is outside the cropped attacking half."""

### 4.2 `render_gaussian`

Renders an isotropic 2-D Gaussian into a `(grid_h, grid_w)` array. Cell `(h, w)` is centered at world `(x_offset + w + 0.5, h + 0.5)`. Centers outside the grid are handled implicitly — only the in-grid portion is kept.

In [ ]:
def render_gaussian(grid_h, grid_w, center_x, center_y, sigma, x_offset=X_MIN):
    col_centers = np.arange(grid_w) + 0.5 + x_offset
    row_centers = np.arange(grid_h) + 0.5
    dx = col_centers[None, :] - center_x
    dy = row_centers[:, None] - center_y
    d2 = dx ** 2 + dy ** 2
    return np.exp(-d2 / (2.0 * sigma ** 2))

### 4.3 Helpers: identify GK, max-combine multi-player channels

In [ ]:
def _identify_goalkeeper(freeze_rows: pd.DataFrame) -> pd.Series:
    """GK = position_name == 'Goalkeeper' AND teammate == False. Tiebreak: closest to (120, 40)."""
    candidates = freeze_rows[
        (freeze_rows["position_name"] == "Goalkeeper")
        & (~freeze_rows["teammate"].astype(bool))
    ]
    if len(candidates) == 0:
        raise ValueError("No defending goalkeeper found in freeze frame.")
    if len(candidates) == 1:
        return candidates.iloc[0]
    gx, gy = GOAL_CENTER
    d2 = (candidates["x"] - gx) ** 2 + (candidates["y"] - gy) ** 2
    return candidates.loc[d2.idxmin()]


def _max_combine(rows: pd.DataFrame) -> np.ndarray:
    """Render each row's Gaussian, combine by elementwise max."""
    out = np.zeros((GRID_H, GRID_W), dtype=np.float64)
    for _, row in rows.iterrows():
        g = render_gaussian(GRID_H, GRID_W, float(row["x"]), float(row["y"]), SIGMA)
        np.maximum(out, g, out=out)
    return out

### 4.4 `rasterize_shot`

In [ ]:
def rasterize_shot(shot_row: pd.Series, freeze_rows: pd.DataFrame) -> torch.Tensor:
    """Rasterize one shot + freeze frame into a (5, 80, 60) float tensor."""
    sx, sy = float(shot_row["x"]), float(shot_row["y"])
    if not (X_MIN <= sx <= X_MAX and Y_MIN <= sy <= Y_MAX):
        raise ShotOutsideCropError(
            f"Shot {shot_row.get('id', '?')} shooter at ({sx:.1f}, {sy:.1f}) "
            f"is outside crop x[{X_MIN},{X_MAX}] y[{Y_MIN},{Y_MAX}]."
        )

    shooter = render_gaussian(GRID_H, GRID_W, sx, sy, SIGMA)
    ball = shooter.copy()  # TODO: pass-end for headers / first-time shots

    gk_row = _identify_goalkeeper(freeze_rows)
    gk = render_gaussian(GRID_H, GRID_W, float(gk_row["x"]), float(gk_row["y"]), SIGMA)

    teammate_mask = freeze_rows["teammate"].astype(bool)
    shooter_pid = shot_row.get("player_id", None)
    teammates_df = freeze_rows[teammate_mask]
    if shooter_pid is not None and not pd.isna(shooter_pid):
        teammates_df = teammates_df[teammates_df["player_id"] != shooter_pid]
    teammates = _max_combine(teammates_df)

    defenders_df = freeze_rows[(~teammate_mask) & (freeze_rows.index != gk_row.name)]
    defenders = _max_combine(defenders_df)

    stack = np.stack([shooter, ball, teammates, defenders, gk], axis=0)
    return torch.from_numpy(stack).float()

### 4.5 `filter_rasterizable_shots`

Idempotent filter: walk a list of shot IDs, drop any whose rasterization fails (off-crop shooter, missing GK, missing freeze entry). The training loop calls this at startup so manifests stay clean.

In [ ]:
def filter_rasterizable_shots(shot_ids, shots_df, freeze_df):
    shots_by_id = shots_df.set_index("id")
    freeze_by_id = freeze_df.groupby("id")
    kept, dropped = [], []
    for sid in shot_ids:
        try:
            shot_row = shots_by_id.loc[sid]
            freeze_rows = freeze_by_id.get_group(sid)
        except KeyError as e:
            dropped.append((sid, f"KeyError: {e}"))
            continue
        try:
            rasterize_shot(shot_row, freeze_rows)
        except (ShotOutsideCropError, ValueError) as e:
            dropped.append((sid, f"{type(e).__name__}: {e}"))
            continue
        kept.append(sid)
    return kept, dropped

### 4.6 Visual sanity check: 5 panels for one training shot

In [ ]:
shots_df_full = pd.read_csv(SHOTS_PATH)
freeze_df_full = pd.read_csv(FREEZE_PATH)
train_ids = pd.read_csv(SPLITS_DIR / "train_shot_ids.csv")["id"].tolist()
shots_by_id = shots_df_full.set_index("id")
freeze_by_id = freeze_df_full.groupby("id")

rng = random.Random(0)
sample_id = None
for sid in [train_ids[0]] + rng.sample(train_ids, 50):
    try:
        t = rasterize_shot(shots_by_id.loc[sid], freeze_by_id.get_group(sid))
    except (ShotOutsideCropError, ValueError, KeyError):
        continue
    sample_id = sid
    break
assert sample_id is not None, "Could not rasterize any of 51 candidate shots."

titles = ["shooter", "ball", "teammates", "defenders", "goalkeeper"]
arr = t.numpy()
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for ax, ch, title in zip(axes, arr, titles):
    ax.imshow(ch, origin="lower", extent=(X_MIN, X_MAX, Y_MIN, Y_MAX),
              vmin=0.0, vmax=1.0, cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
fig.suptitle(f"Shot {sample_id}")
fig.tight_layout()
plt.show()

### 4.7 Validate on 5 random training shots

In [ ]:
sample = rng.sample(train_ids, 5)
print("Validation on 5 random training shots:")
for sid in sample:
    try:
        t = rasterize_shot(shots_by_id.loc[sid], freeze_by_id.get_group(sid))
    except (ShotOutsideCropError, ValueError, KeyError) as e:
        print(f"  {sid}: SKIPPED ({type(e).__name__}: {e})")
        continue
    assert t.shape == (5, GRID_H, GRID_W), f"bad shape {tuple(t.shape)} for {sid}"
    assert not torch.isnan(t).any(), f"NaN in {sid}"
    assert (t >= 0.0).all() and (t <= 1.0).all(), f"out-of-range in {sid}"
    print(f"  {sid}: OK shape={tuple(t.shape)} min={t.min():.3f} max={t.max():.3f}")

## 5. PyTorch `Dataset` and `DataLoader`s

`GoalkeeperShotsDataset` returns `(raster_tensor, is_goal)` per item. With `cache_in_memory=True`, all shots are rasterized at `__init__` — any rasterization failures surface immediately rather than mid-epoch.

The training set caches in memory (~2.4 GB for ~30k shots × 96 KB/shot). The transfer splits are not part of the standard training pipeline — they're held out and only loaded ad hoc for post-hoc evaluation.

### 5.1 `GoalkeeperShotsDataset`

In [ ]:
BYTES_PER_TENSOR = 5 * GRID_H * GRID_W * 4  # 5 channels × 80 × 60 × float32


class GoalkeeperShotsDataset(Dataset):
    """One split of shots → rasterized tensors and is_goal labels."""

    def __init__(self, manifest_path, shots_df, freeze_df, cache_in_memory=False):
        manifest = pd.read_csv(manifest_path)
        missing = {"id", "is_goal"} - set(manifest.columns)
        if missing:
            raise ValueError(
                f"Manifest {manifest_path} missing required columns {missing}; "
                f"got {list(manifest.columns)}."
            )
        self.shot_ids = manifest["id"].tolist()
        self.labels = torch.tensor(
            manifest["is_goal"].astype(float).values, dtype=torch.float32
        )
        self._shots_by_id = shots_df.set_index("id")
        self._freeze_by_id = freeze_df.groupby("id")

        self._cache = None
        if cache_in_memory:
            n = len(self.shot_ids)
            est_mb = n * BYTES_PER_TENSOR / (1024 * 1024)
            print(f"GoalkeeperShotsDataset({manifest_path.name}): caching {n} shots "
                  f"in memory ({BYTES_PER_TENSOR / 1024:.1f} KB/shot, ~{est_mb:.0f} MB total).")
            self._cache = [self._rasterize(i) for i in range(n)]

    def _rasterize(self, idx):
        sid = self.shot_ids[idx]
        return rasterize_shot(self._shots_by_id.loc[sid], self._freeze_by_id.get_group(sid))

    def __len__(self):
        return len(self.shot_ids)

    def __getitem__(self, idx):
        x = self._cache[idx] if self._cache is not None else self._rasterize(idx)
        return x, self.labels[idx]

### 5.2 `make_dataloaders`

In [ ]:
def make_dataloaders(batch_size=64, num_workers=4, cache_in_memory=False):
    """Build train/val/test DataLoaders sharing a single load of the master CSVs.

    Train shuffles; val and test do not. pin_memory and persistent_workers are
    enabled when num_workers > 0 to keep training-loop overhead low.
    """
    shots = pd.read_csv(SHOTS_PATH)
    freeze = pd.read_csv(FREEZE_PATH)

    paths = {
        "train": SPLITS_DIR / "train_shot_ids.csv",
        "val":   SPLITS_DIR / "val_shot_ids.csv",
        "test":  SPLITS_DIR / "test_shot_ids.csv",
    }
    datasets = {
        name: GoalkeeperShotsDataset(p, shots, freeze, cache_in_memory=cache_in_memory)
        for name, p in paths.items()
    }
    common = {
        "batch_size": batch_size,
        "num_workers": num_workers,
        "pin_memory": torch.cuda.is_available(),
        "persistent_workers": num_workers > 0,
    }
    return {
        "train": DataLoader(datasets["train"], shuffle=True, **common),
        "val":   DataLoader(datasets["val"], shuffle=False, **common),
        "test":  DataLoader(datasets["test"], shuffle=False, **common),
    }

### 5.3 Smoke test

Build all three loaders without caching and pull one batch from each. `num_workers=0` so any rare unfiltered off-crop shot surfaces here rather than in a worker process.

In [ ]:
loaders = make_dataloaders(batch_size=8, num_workers=0, cache_in_memory=False)
for name, loader in loaders.items():
    x, y = next(iter(loader))
    print(f"{name}: dataset_size={len(loader.dataset)} "
          f"batch x={tuple(x.shape)} y={tuple(y.shape)} "
          f"label_mean={y.mean().item():.3f}")

---

**Done.** Splits are written to `data/processed/splits/` and the data primitives (`rasterize_shot`, `filter_rasterizable_shots`, `GoalkeeperShotsDataset`, `make_dataloaders`) are defined and verified.

The training notebook (`02_model_and_training.ipynb`) imports them from `src/data/` and `src/training/` — those modules can either be kept as a mirror of the code here or removed once the team agrees notebooks are the canonical source. For now we keep both so the existing CLI scripts (`python src/training/train.py`) still work.